In [ ]:
import sys
from pathlib import Path
from transformers import ViTImageProcessor

# Resolve project root directory from notebook location
notebook_path = Path.cwd()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.evaluation.vision_eval import VisionNeglectEvaluator
from src.models.loader import ModelEnvironmentLoader
from utils.memory import flush_memory, print_vram_usage

print("Vision environment setup completed.")

In [ ]:
# Configuration block targeting minimal VRAM footprint for ViT
vision_config = {
    "model": {
        "model_name": "google/vit-base-patch16-224",
        "torch_dtype": "bfloat16",
        "device": "cuda",
    }
}

model_name = vision_config["model"]["model_name"]
loader = ModelEnvironmentLoader(vision_config)

# Load base classification model and matching image processor
vit_model = loader.load_vision_model(model_name)
vit_processor = ViTImageProcessor.from_pretrained(model_name)

evaluator = VisionNeglectEvaluator(vit_model, vit_processor)
print_vram_usage()

In [ ]:
# Setup placeholder sample path (Ensure an image exists at this location before running)
sample_image_path = project_root / "data" / "vision_samples" / "sample.jpg"

print("--- Running Baseline Vision Evaluation ---")
try:
    baseline_result = evaluator.evaluate_image(sample_image_path)
except FileNotFoundError:
    print(f"Please place a sample image at: {sample_image_path}")

In [ ]:
print("--- Simulating Sensory Neglect via Physical Input Masking ---")
try:
    masked_result = evaluator.evaluate_image(sample_image_path, mask_side="left")
    print(f"Shift in prediction confidence: {masked_result['confidence']:.4f}")
except FileNotFoundError:
    print("Aborting physical mask test due to missing sample image.")

In [ ]:
print("--- Injecting Cognitive Neglect Hook into ViT self-attention ---")


def vit_spatial_neglect_hook(module, input_tensor, output_tensor):
    """Corrupt spatial token self-attention matrices during forward pass."""
    # ViT attention weights shape usually: [batch, heads, sequence_len, sequence_len]
    # Forcing a subset of head connections to zero to blind specific patch grids
    if isinstance(output_tensor, tuple):
        matrix = output_tensor[0]
        matrix[:, :, :, :10] = 0.0
        return (matrix,) + output_tensor[1:]
    else:
        output_tensor[:, :, :, :10] = 0.0
        return output_tensor


# Attach hook to the first encoder block self-attention layer
target_layer = vit_model.vit.encoder.layer[0].attention.attention
hook_handle = target_layer.register_forward_hook(vit_spatial_neglect_hook)
print("Mechanistic neural lesion hook is now active.")

try:
    mechanistic_result = evaluator.evaluate_image(sample_image_path)
finally:
    # Always remove the hook to prevent persistent state pollution
    hook_handle.remove()
    print("Mechanistic vision hook removed safely.")

In [ ]:
print("Freeing vision model components and clearing GPU cache...")
del vit_model
del vit_processor
del evaluator

flush_memory()
print_vram_usage()